In [0]:
%sql
use catalog ext_cat

In [0]:
%sql
drop table if exists employees;
create table employees;

Table has no columns, but the file has columns, so we need to specify copy_options - mergeschema, so that the table gets altered to match the incoming file

In [0]:
%sql
copy into employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'mergeSchema'='true',
    'delimiter' = ','
)
copy_options(
    'mergeSchema' = 'true'
)

 

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
10,10,0


All columns are string

In [0]:
%sql
select * from employees

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID
1,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600,-,124,50
2,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600,-,124,50
3,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10
4,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20
5,Pat,Fay,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20
6,Susan,Mavris,SMAVRIS,515.123.7777,07-Jun-02,HR_REP,6500,-,101,40
7,Hermann,Baer,HBAER,515.123.8888,07-Jun-02,PR_REP,10000,-,101,70
8,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008,-,101,110
9,William,Gietz,WGIETZ,515.123.8181,07-Jun-02,AC_ACCOUNT,8300,-,205,110
10,Steven,King,SKING,515.123.4567,17-Jun-03,AD_PRES,24000,-,-,90


Run the copyinto again

In [0]:
%sql
copy into employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'mergeSchema'='true',
    'delimiter' = ','
)
copy_options(
    'mergeSchema' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


Though we ran, nothing is processed. as the source file is already processed. This is the speciality of `copy into` compared to `CTAS`  - its idempotent

Loaded a new file to source location. now re running copy into

In [0]:
%sql
copy into employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'mergeSchema'='true',
    'delimiter' = ','
)
copy_options(
    'mergeSchema' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
10,10,0


In [0]:
%sql
select * from employees

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,BONUS
51,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600,-,124,None,1
52,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600,-,124,None,1
53,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,Zero,1
54,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20,1
55,Pat,Fay,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20,1
56,Susan,Mavris,SMAVRIS,515.123.7777,07-Jun-02,HR_REP,6500,-,101,40,1
57,Hermann,Baer,HBAER,515.123.8888,07-Jun-02,PR_REP,10000,-,101,70,1
58,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008,-,101,110,1
59,William,Gietz,WGIETZ,515.123.8181,07-Jun-02,AC_ACCOUNT,8300,-,205,110,1
60,Steven,King,SKING,515.123.4567,17-Jun-03,AD_PRES,24000,-,-,90,1


new file is loaded now which had one extra column. Due to copyoption-mergeschema coded, it accepted the new file, and automatically altered the table to have new column

Schema has evolved, as the new file had an extra column
with out the use of  copy_options(     'mergeSchema' = 'true' )  this will  fail

In [0]:
%sql
describe history ext_cat.default.employees

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-05T08:47:44Z,147836707444603,anooptu@gmail.com,COPY INTO,Map(statsOnLoad -> false),null,List(4218852656760406),0826-063441-ydade36q,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 10, numOutputBytes -> 4483, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.13
1,2026-09-05T08:46:25Z,147836707444603,anooptu@gmail.com,COPY INTO,Map(statsOnLoad -> false),null,List(4218852656760406),0826-063441-ydade36q,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 10, numOutputBytes -> 4217, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.13
0,2026-09-05T08:45:54Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(4218852656760406),0826-063441-ydade36q,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13


Dropping and recreating for next demo

In [0]:
%sql
drop table if exists employees;
create table employees;

In [0]:
%sql
copy into employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'inferSchema'='true',
    'mergeSchema'='true',
    'delimiter' = ','
)
copy_options(
    'mergeSchema' = 'true'
)

 

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
10,10,0


Columns with Numeric values are now inferred as int, when    'inferSchema'='true', was mentioned

In [0]:
%sql
describe formatted employees;

col_name,data_type,comment
EMPLOYEE_ID,int,null
FIRST_NAME,string,null
LAST_NAME,string,null
EMAIL,string,null
PHONE_NUMBER,string,null
HIRE_DATE,string,null
JOB_ID,string,null
SALARY,int,null
COMMISSION_PCT,string,null
MANAGER_ID,string,null


Now if we upload another file with exact schema, copy_options - merge schema is not required as the schema is matching

In [0]:
%sql
copy into employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'inferSchema'='true',
    'delimiter' = ','
)
 

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
10,10,0


In [0]:
%sql
describe formatted employees;

col_name,data_type,comment
EMPLOYEE_ID,int,null
FIRST_NAME,string,null
LAST_NAME,string,null
EMAIL,string,null
PHONE_NUMBER,string,null
HIRE_DATE,string,null
JOB_ID,string,null
SALARY,int,null
COMMISSION_PCT,string,null
MANAGER_ID,string,null


Uploading another file with different schema (DEPARTMENT_ID has string values) with out merge schema

In [0]:
%sql
copy into employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'inferSchema'='true',    
    'mergeSchema'='true',
    'delimiter' = ','
)
 

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4515921983465338>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'copy into employees\nfrom \'/Volumes/ext_cat/default/extvol/emp/\'\nfileformat = CSV\nformat_options(\n    "header"="true",\n    \'inferSchema\'=\'true\',    \n    \'mergeSchema\'=\'true\',\n    \'delimiter\' = \',\'\n)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):


Retry with copy_options(     'mergeSchema' = 'true' )

In [0]:
%sql
copy into ext_cat.default.employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'inferSchema'='true',    
    'mergeSchema'='true',
    'delimiter' = ','
)
 copy_options(
    'mergeSchema' = 'true'
)


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5849574006095900>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'copy into ext_cat.default.employees\nfrom \'/Volumes/ext_cat/default/extvol/emp/\'\nfileformat = CSV\nformat_options(\n    "header"="true",\n    \'mergeSchema\'=\'true\',\n    \'delimiter\' = \',\'\n)\n copy_options(\n    \'mergeSchema\' = \'true\'\n)\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_

copy_options('mergeSchema'='true') can add new columns and widen compatible numeric types (e.g., int → long), but it cannot change an existing column's type from int to string — those are incompatible in Delta. In the new file uploaded we had string data to be loaded to int column 

corrected the file and retyring

In [0]:
%sql
copy into ext_cat.default.employees
from '/Volumes/ext_cat/default/extvol/emp/'
fileformat = CSV
format_options(
    "header"="true",
    'inferSchema'='true',  
    'mergeSchema'='true',
    'delimiter' = ','
)
 copy_options(
    'mergeSchema' = 'true'
)


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
10,10,0


new column in the new file added to the table

In [0]:
%sql
select * from ext_cat.default.employees

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,BONUS
51,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600,-,124,1,1
52,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600,-,124,1,1
53,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,1,1
54,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20,1
55,Pat,Fay,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20,1
56,Susan,Mavris,SMAVRIS,515.123.7777,07-Jun-02,HR_REP,6500,-,101,40,1
57,Hermann,Baer,HBAER,515.123.8888,07-Jun-02,PR_REP,10000,-,101,70,1
58,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008,-,101,110,1
59,William,Gietz,WGIETZ,515.123.8181,07-Jun-02,AC_ACCOUNT,8300,-,205,110,1
60,Steven,King,SKING,515.123.4567,17-Jun-03,AD_PRES,24000,-,-,90,1
